In [3]:
import pandas as pd
import numpy as np
import seaborn as sns

In [4]:
df_links = pd.read_csv('links.csv')
df_movies = pd.read_csv('movies.csv')
df_ratings = pd.read_csv('ratings.csv')
df_tags = pd.read_csv('tags.csv')


In [5]:
def head(df):
    return df.head()

def isnull(df):
    return df.sum().isnull()


print(head(df_links))
print(head(df_movies))
print(head(df_ratings))
head(df_tags)

   movieId  imdbId   tmdbId
0        1  114709    862.0
1        2  113497   8844.0
2        3  113228  15602.0
3        4  114885  31357.0
4        5  113041  11862.0
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931


,userId,movieId,tag,timestamp
0,2,60756,funny,1445714994
1,2,60756,Highly quotable,1445714996
2,2,60756,will ferrell,1445714992
3,2,89774,Boxing story,1445715207
4,2,89774,MMA,1445715200


In [6]:
print(isnull(df_links))
print(isnull(df_movies))
print(isnull(df_ratings))
print(isnull(df_tags))

movieId    False
imdbId     False
tmdbId     False
dtype: bool
movieId    False
title      False
genres     False
dtype: bool
userId       False
movieId      False
rating       False
timestamp    False
dtype: bool
userId       False
movieId      False
tag          False
timestamp    False
dtype: bool


In [7]:
pip install tmdbv3api

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
from tmdbv3api import TMDb, Movie
from tqdm import tqdm
tmdb = TMDb()
tmdb.api_key = '7638ae832eab6400fbb78b923429bcb8'
tmdb.language = 'fr'
movie_api = Movie()
enriched_data = []

first_valid_row = df_links.dropna(subset=['tmdbId']).iloc[0]
tmdb_id = int(first_valid_row['tmdbId'])
m = movie_api.details(int(tmdb_id))
print(m.__dict__.keys())

dict_keys(['_json', '_key', '_dict_key', '_dict_key_name', '_obj_list', '_list_only', 'adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count', 'videos', 'trailers', 'images', 'casts', 'translations', 'keywords', 'release_dates'])


In [ ]:
"""""
#on enrichit les données avec l'API TMDb pour avoir du contexte.
tmdb = TMDb()
tmdb.api_key = '7638ae832eab6400fbb78b923429bcb8'
tmdb.language = 'fr'
movie_api = Movie()
enriched_data = []

for index, row in tqdm(df_links.iterrows(), total = df_links.shape[0]):
    tmdb_id = row['tmdbId']
    
    if pd.isna(tmdb_id):
        enriched_data.append({})
        continue
    
    try:
        m = movie_api.details(int(tmdb_id))
        #print(m.__dict__)
        #print(m.entries.keys())
        enriched_data.append({
            'movieId': row['movieId'], 
            'budget': getattr(m, 'budget', 0),
            'revenue': getattr(m, 'revenue', 0),
            'runtime': getattr(m, 'runtime', 0),
            'release_date': getattr(m, 'release_date', np.nan),
            'vote_average_tmdb': getattr(m, 'vote_average', 0),
            'vote_count_tmdb': getattr(m, 'vote_count', 0)
        })
        
    except Exception as e:
        enriched_data.append({'movieId': row['movieId']})
        
df_enriched = pd.DataFrame(enriched_data)
df_enriched.to_csv('movies_enriched.csv', index = False)
df_meta = pd.read_csv('movies_enriched.csv')
"""

100%|██████████| 9742/9742 [2:05:50<00:00,  1.29it/s]  


In [ ]:
df_final = df_movies.merge(df_meta, on = 'movieId', how = 'left')
print(df_final.head())